In [ ]:
!nvidia-smi

Wed Jul 15 13:59:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install -U ultralytics==8.4.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.7 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# paths
ZIP_PATH = "/content/drive/MyDrive/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph.zip"
WORK_DIR = "/content"

# copy & unzip
!cp "$ZIP_PATH" $WORK_DIR/data.zip
!unzip -o $WORK_DIR/data.zip -d $WORK_DIR

Archive:  /content/data.zip
   creating: /content/Dataset/
  inflating: /content/Dataset/classes.txt  
   creating: /content/Dataset/images/
  inflating: /content/Dataset/images/00001.jpg  
  inflating: /content/Dataset/images/000010.jpg  
  inflating: /content/Dataset/images/0000100.jpg  
  inflating: /content/Dataset/images/00001000.jpg  
  inflating: /content/Dataset/images/00001001.jpg  
  inflating: /content/Dataset/images/00001002.jpg  
  inflating: /content/Dataset/images/00001003.jpg  
  inflating: /content/Dataset/images/00001004.jpg  
  inflating: /content/Dataset/images/00001005.jpg  
  inflating: /content/Dataset/images/00001006.jpg  
  inflating: /content/Dataset/images/00001007.jpg  
  inflating: /content/Dataset/images/00001008.jpg  
  inflating: /content/Dataset/images/00001009.jpg  
  inflating: /content/Dataset/images/0000101.jpg  
  inflating: /content/Dataset/images/00001010.jpg  
  inflating: /content/Dataset/images/00001011.jpg  
  inflating: /content/Dataset/imag

In [ ]:
import glob, os, random, shutil

random.seed(42)

SRC_ROOT = "/content/Dataset"          # root containing your extracted images (+ labels, if detection)
SPLIT_ROOT = "/content/Data_split"  # new root with clean train/val/test folders

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

# Recursively find every image under SRC_ROOT, wherever it lives in the extracted zip
all_images = [p for p in glob.glob(os.path.join(SRC_ROOT, "**", "*"), recursive=True)
              if p.lower().endswith(IMG_EXTS)]
assert len(all_images) > 0, f"No images found under {SRC_ROOT} — check ZIP_PATH extraction / folder names."
print(f"Found {len(all_images)} images under {SRC_ROOT}")

# Try to find a matching YOLO-format label (.txt) for each image, if this is a detection task.
# Label is assumed to sit in a parallel 'labels' folder with the same filename stem.
def find_label(img_path):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    candidates = glob.glob(os.path.join(SRC_ROOT, "**", stem + ".txt"), recursive=True)
    # Prefer a candidate that lives under a "labels" folder if there are multiple hits
    labels_only = [c for c in candidates if os.sep + "labels" in c]
    return (labels_only or candidates or [None])[0]

pairs = [(img, find_label(img)) for img in all_images]
n_with_labels = sum(1 for _, l in pairs if l)
print(f"{n_with_labels}/{len(pairs)} images have a matching label file "
      f"({'detection task' if n_with_labels else 'no labels found — treating as classification/no-label set'})")

random.shuffle(pairs)
n = len(pairs)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
splits = {
    "train": pairs[:n_train],
    "val": pairs[n_train:n_train + n_val],
    "test": pairs[n_train + n_val:],
}
for name, s in splits.items():
    print(f"{name}: {len(s)} images")

for split_name, split_pairs in splits.items():
    img_dir = os.path.join(SPLIT_ROOT, "images", split_name)
    lbl_dir = os.path.join(SPLIT_ROOT, "labels", split_name)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    for img_path, lbl_path in split_pairs:
        shutil.copy2(img_path, os.path.join(img_dir, os.path.basename(img_path)))
        if lbl_path:
            shutil.copy2(lbl_path, os.path.join(lbl_dir, os.path.basename(lbl_path)))

print("Split complete. No image appears in more than one of train/val/test.")


Found 2095 images under /content/Dataset
2095/2095 images have a matching label file (detection task)
train: 1676 images
val: 209 images
test: 210 images
Split complete. No image appears in more than one of train/val/test.


In [ ]:
import glob

bad_files = []

for f in glob.glob("/content/Data_split/labels/**/*.txt", recursive=True):
    with open(f) as file:
        for line in file:
            cls = int(float(line.split()[0]))
            if cls > 6:
                bad_files.append((f, cls))

print("Bad label files:", len(bad_files))

for f, cls in bad_files[:20]:
    print(cls, f)

Bad label files: 613
7 /content/Data_split/labels/test/0000707.txt
7 /content/Data_split/labels/test/0000707.txt
7 /content/Data_split/labels/test/00001038.txt
7 /content/Data_split/labels/test/00001038.txt
7 /content/Data_split/labels/test/00001038.txt
7 /content/Data_split/labels/test/00001254.txt
7 /content/Data_split/labels/test/00001254.txt
7 /content/Data_split/labels/test/00001254.txt
7 /content/Data_split/labels/test/nir53.txt
7 /content/Data_split/labels/test/nir53.txt
7 /content/Data_split/labels/test/0000563.txt
7 /content/Data_split/labels/test/nir12.txt
7 /content/Data_split/labels/test/nir12.txt
8 /content/Data_split/labels/test/ibn181.txt
8 /content/Data_split/labels/test/ibn181.txt
7 /content/Data_split/labels/test/00001439.txt
8 /content/Data_split/labels/test/ibn154.txt
8 /content/Data_split/labels/test/ibn154.txt
8 /content/Data_split/labels/test/ibn154.txt
8 /content/Data_split/labels/test/ibn281.txt


In [ ]:
import os
import glob

LABEL_ROOT = "/content/Data_split/labels"

valid_classes = set(range(7))   # Keep only classes 0-6

removed_boxes = 0
modified_files = 0
empty_files = []

label_files = glob.glob(os.path.join(LABEL_ROOT, "**", "*.txt"), recursive=True)

for label_file in label_files:
    with open(label_file, "r") as f:
        lines = f.readlines()

    new_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        parts = line.split()

        try:
            cls = int(float(parts[0]))
        except:
            continue

        # Keep only valid classes
        if cls in valid_classes:
            new_lines.append(line + "\n")
        else:
            removed_boxes += 1

    if len(new_lines) != len(lines):
        modified_files += 1

    with open(label_file, "w") as f:
        f.writelines(new_lines)

    if len(new_lines) == 0:
        empty_files.append(label_file)

print("="*60)
print("Cleaning completed")
print("="*60)
print(f"Total label files scanned : {len(label_files)}")
print(f"Modified label files      : {modified_files}")
print(f"Invalid annotations removed: {removed_boxes}")
print(f"Empty label files         : {len(empty_files)}")
print("="*60)

if empty_files:
    print("\nFirst 20 empty label files:")
    for f in empty_files[:20]:
        print(f)

Cleaning completed
Total label files scanned : 2095
Modified label files      : 343
Invalid annotations removed: 613
Empty label files         : 20

First 20 empty label files:
/content/Data_split/labels/test/0000682.txt
/content/Data_split/labels/train/ibn395.txt
/content/Data_split/labels/train/ibn479.txt
/content/Data_split/labels/train/00001373.txt
/content/Data_split/labels/train/ibn347.txt
/content/Data_split/labels/train/00001120.txt
/content/Data_split/labels/train/00001480.txt
/content/Data_split/labels/train/nir71.txt
/content/Data_split/labels/train/ibn524.txt
/content/Data_split/labels/train/ibn378.txt
/content/Data_split/labels/train/ibn306.txt
/content/Data_split/labels/train/0000264.txt
/content/Data_split/labels/train/0000818.txt
/content/Data_split/labels/train/0000737.txt
/content/Data_split/labels/train/0000179.txt
/content/Data_split/labels/train/ibn88.txt
/content/Data_split/labels/train/ibn231.txt
/content/Data_split/labels/train/ibn322.txt
/content/Data_split/lab

In [ ]:
import glob

bad_files = []

for f in glob.glob("/content/Data_split/labels/**/*.txt", recursive=True):
    with open(f) as file:
        for line in file:
            cls = int(float(line.split()[0]))
            if cls > 6:
                bad_files.append((f, cls))

print("Bad label files:", len(bad_files))

for f, cls in bad_files[:20]:
    print(cls, f)

Bad label files: 0


In [ ]:
classes_path = "/content/Dataset/classes.txt"
yaml_path = "/content/data.yaml"

with open(classes_path) as f:
    classes = [c.strip() for c in f if c.strip()]

with open(yaml_path, "w") as f:
    f.write(f"path: {SPLIT_ROOT}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("names:\n")
    for i, c in enumerate(classes):
        f.write(f"  {i}: {c}\n")

print("Classes:", classes)
print(open(yaml_path).read())


Classes: ['Missing', 'Dental Crown', 'Root Canal', 'Caries', 'Broken Down', 'Wisdom Teeth', 'Healthy']
path: /content/Data_split
train: images/train
val: images/val
names:
  0: Missing
  1: Dental Crown
  2: Root Canal
  3: Caries
  4: Broken Down
  5: Wisdom Teeth
  6: Healthy



In [ ]:
!wget -q https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11m.pt
assert os.path.exists("yolo11m.pt"), "YOLOv11 weights missing!"


In [ ]:
model = YOLO("yolo11m.pt")
model.model.info()


YOLO11m summary: 231 layers, 20,114,688 parameters, 0 gradients, 68.5 GFLOPs


(231, 20114688, 0, 68.52838399999999)

In [ ]:
results = model.train(
    model="yolo11m.pt",
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    box=9.0,
    dfl=2.0,
    cls=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=5.0,
    scale=0.5,
    project="runs/scale_adaptive",
    name="sa_yolov11m"
)


New https://pypi.org/project/ultralytics/8.4.96 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=9.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sa_yolov11m, nbs=64, nms=Fals

In [ ]:
metrics_val_v11 = model.val(
    data="/content/data.yaml",
    split="val",
    imgsz=640,
    device=0
)

print("=== YOLOv11m VALIDATION RESULTS ===")
print("mAP@0.5     :", metrics_val_v11.box.map50)
print("mAP@0.5:0.95:", metrics_val_v11.box.map)
print("Precision   :", metrics_val_v11.box.mp)
print("Recall      :", metrics_val_v11.box.mr)

Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11m summary (fused): 126 layers, 20,035,429 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3421.9±1373.9 MB/s, size: 845.8 KB)
val: Scanning /content/Data_split/labels/val.cache... 209 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 209/209 87.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 4.5it/s 3.1s
                   all        209        918      0.746      0.764      0.769      0.368
               Missing        104        240      0.571        0.5      0.458      0.133
          Dental Crown         62        192      0.942      0.948      0.968      0.521
            Root Canal         69        187      0.722      0.737      0.679      0.214
                Caries         84        127      0.698      0.591      0.659      0.259
           Broken Down       

In [ ]:
model_v10 = YOLO("yolov10m.pt")  # auto-downloads official YOLOv10n weights
model_v10.model.info()


YOLOv10m summary: 288 layers, 16,576,768 parameters, 0 gradients, 64.5 GFLOPs


(288, 16576768, 0, 64.4772096)

In [ ]:
results = model_v10.train(
    model="yolov10m.pt",
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    box=9.0,
    dfl=2.0,
    cls=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=5.0,
    scale=0.5,
    project="runs/scale_adaptive",
    name="sa_yolov10m"
)

New https://pypi.org/project/ultralytics/8.4.96 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=9.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov10m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sa_yolov10m, nbs=64, nms=Fal

In [ ]:
metrics_val_v10 = model_v10.val(
    data="/content/data.yaml",
    split="val",
    imgsz=640,
    device=0
)

print("=== YOLOv10m VALIDATION RESULTS ===")
print("mAP@0.5     :", metrics_val_v10.box.map50)
print("mAP@0.5:0.95:", metrics_val_v10.box.map)
print("Precision   :", metrics_val_v10.box.mp)
print("Recall      :", metrics_val_v10.box.mr)


Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv10m summary (fused): 136 layers, 15,317,221 parameters, 0 gradients, 58.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3872.2±1376.2 MB/s, size: 950.4 KB)
val: Scanning /content/Data_split/labels/val.cache... 209 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 209/209 79.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 5.0it/s 2.8s
                   all        209        918      0.728      0.759      0.759       0.36
               Missing        104        240      0.529      0.521      0.452      0.136
          Dental Crown         62        192       0.91      0.974      0.968      0.517
            Root Canal         69        187       0.67       0.77      0.712      0.224
                Caries         84        127      0.655      0.614      0.614      0.235
           Broken Down      

In [ ]:
model_v8 = YOLO("yolov8m.pt")  # auto-downloads official YOLOv8m weights
model_v8.model.info()

YOLOv8m summary: 169 layers, 25,902,640 parameters, 0 gradients, 79.3 GFLOPs


(169, 25902640, 0, 79.3204224)

In [ ]:
results_v12 = model_v8.train(
    model="yolo8m.pt",
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    box=9.0,
    dfl=2.0,
    cls=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=5.0,
    scale=0.5,
    project="runs/scale_adaptive",
    name="sa_yolov8"
)


New https://pypi.org/project/ultralytics/8.4.96 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=9.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sa_yolov8, nbs=64, nms=False, 

In [ ]:
metrics_val_v8 = model_v8.val(
    data="/content/data.yaml",
    split="val",
    imgsz=640,
    device=0
)

print("=== YOLOv8m VALIDATION RESULTS ===")
print("mAP@0.5     :", metrics_val_v10.box.map50)
print("mAP@0.5:0.95:", metrics_val_v10.box.map)
print("Precision   :", metrics_val_v10.box.mp)
print("Recall      :", metrics_val_v10.box.mr)


Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 93 layers, 25,843,813 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2113.0±1276.7 MB/s, size: 950.4 KB)
val: Scanning /content/Data_split/labels/val.cache... 209 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 209/209 73.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 4.5it/s 3.1s
                   all        209        918      0.694      0.755      0.747      0.354
               Missing        104        240      0.468      0.467      0.416      0.125
          Dental Crown         62        192      0.953      0.954      0.987      0.531
            Root Canal         69        187      0.679       0.69      0.621      0.186
                Caries         84        127      0.664      0.544      0.585      0.222
           Broken Down         2

In [ ]:
import shutil

zip_path = "/content/runs.zip"

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir="/content",
    base_dir="runs"
)

print("✅ runs folder zipped at:", zip_path)


✅ runs folder zipped at: /content/runs.zip
